In [4]:
import os
import json
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage


load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")

In [14]:
system_prompt = """
# 身份
- 你是一个编程专家。

# 指令
- 定义变量时，使用snake case命名法，而不是camel case命名法。
- 不要返回markdown格式说明，仅仅返回代码即可。

"""

agent = create_agent(model="deepseek-v4-pro",
                     system_prompt=system_prompt)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="怎样定义string变量记录学校名字，例如`程序员`")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

school_name = "程序员"

In [15]:

system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 示例
user：月球的首都是什么？
assistant：月华城（Lunara）—— 镶嵌在月球静海环形山中的水晶穹顶都市，其核心是一座利用月球潮汐能驱动的巨型生态循环塔。

user：火星的首都是什么？
assistant：赤晶城（Aresia）—— 深嵌于火星奥林匹斯山熔岩管内的蜂巢都市，地表仅露出由火星红土烧制而成的螺旋尖塔。
"""

# 创建智能体

agent = create_agent(model="deepseek-v4-pro",
                     system_prompt=system_prompt)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

辉烬城（Cytherea）—— 悬浮在金星硫酸云层上空的浮空碟城，依靠巨型超导磁环对抗地表地狱般的压强与高温，整座城市笼罩在金色磷光护盾中，宛如一颗永恒燃烧的琥珀。

In [16]:

system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 指令
- 请务必以JSON格式输出，不要加任何markdown样式。

# 示例：
user: 月球的首都是什么？
assistant:
{
    "name": "月华市（Lunaria）",
    "location": "位于月球正面赤道附近的静海基地遗址之上，依托巨大的穹顶与地下网络建成",
    "vibe": "冷冽、高效、革新",
    "economy": "氦-3能源开采、量子通信枢纽、尖端生物圈农业"
}
"""

agent = create_agent(model="deepseek-v4-pro",
                     system_prompt=system_prompt)

response = agent.invoke(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
)

print(response['messages'][-1].content)

{
    "name": "辉光城（Glowhaven）",
    "location": "漂浮在金星上层大气中的云洲链，借助耐腐蚀气囊与反重力引擎稳定在50公里高空",
    "vibe": "神秘、浓密、迷幻",
    "economy": "大气层碳提取与合成材料、高能粒子实验、极端环境旅游业"
}


In [25]:
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 首先，我们定义一个类，用来封装模型要输出的数据：
class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

# 创建智能体并设置结构化输出的格式了。
agent = create_agent(
    model="deepseek-chat",
    system_prompt="你是一个科幻作家，根据用户的要求创建一个太空之都。",
    response_format=CapitalInfo # 设置结构化输出的格式
)

response = agent.invoke(
    {"messages": [HumanMessage(content="月球的首都是什么?")]}
)

city = response['structured_response']
print(response)
print(f"{city.name}位于{city.location}，是一座{city.vibe}的城市，其主要产业包括{city.economy}。")

{'messages': [HumanMessage(content='月球的首都是什么?', additional_kwargs={}, response_metadata={}, id='a41f0868-12b3-43c5-81a5-da00d37c74e2'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 330, 'total_tokens': 417, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 74}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'b3901955-55ef-48c0-851b-f0f2077ce1fd', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e6dd7-6a26-7762-b7f3-e9e15b607f37-0', tool_calls=[{'name': 'CapitalInfo', 'args': {'name': '月球首都', 'location': '月球', 'vibe': '未知', 'economy': '未知'}, 'id': 'call_00_0dY9olr2TTbWJFCenunC9109', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 330, 'output_toke